In [2]:
import numpy as np
import matplotlib
#不显示窗口保存图片
matplotlib.use("Agg")

import matplotlib.pyplot as plt

#下面这一段是让中文显示正常
from matplotlib import font_manager
plt.rcParams["font.sans-serif"] = ["SimHei"]  # 步骤 1：直接用 Windows 自带的黑体
plt.rcParams["axes.unicode_minus"] = False     # 步骤 2：保证负号不报错

#自定义颜色
from matplotlib.colors import ListedColormap
#乳腺癌数据
from sklearn.datasets import load_breast_cancer
#K折自动找最好参数
from sklearn.model_selection import train_test_split, GridSearchCV
#标准化
from sklearn.preprocessing import StandardScaler
#流水线避免数据泄露
from sklearn.pipeline import Pipeline
#核心SVM分类
from sklearn.svm import SVC
#主成分分析用于降维然后可视化
from sklearn.decomposition import PCA
#混淆军阵分类报告
from sklearn.metrics import classification_report, confusion_matrix


# ---------- 1) 加载真实数据 ----------
data = load_breast_cancer()
X, y = data.data, data.target          # 569 个样本, 30 个特征, 二分类
print("数据形状:", X.shape, "| 类别:", dict(zip(data.target_names, np.bincount(y))))

# ---------- 2) 划分训练 / 测试 ----------
X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=0)#stratify表示最好让01分布均匀

# ---------- 3) 流水线: 标准化 -> RBF 核 SVM ----------
pipe = Pipeline([("scaler", StandardScaler()),
                 ("svc", SVC(kernel="rbf"))])

# ---------- 4) 网格搜索调 C 和 gamma (5 折交叉验证) ----------
param_grid = {"svc__C":     [0.1, 1, 10, 100],
              "svc__gamma": [1e-4, 1e-3, 1e-2, 1e-1, 1]}
grid = GridSearchCV(pipe, param_grid, cv=5, scoring="accuracy")
grid.fit(X_tr, y_tr)
print("最佳参数:", grid.best_params_, "| 交叉验证准确率: %.3f" % grid.best_score_)

# ---------- 5) 在从未见过的测试集上评估 ----------
y_pred = grid.predict(X_te)
print("\n测试集表现:\n", classification_report(y_te, y_pred, target_names=data.target_names))

# ================= 可视化 =================
blue, coral = "#378ADD", "#D85A30"
fig, axes = plt.subplots(1, 3, figsize=(15.5, 4.6))

# (a) 网格搜索热力图: (gamma, C) -> 交叉验证准确率
C_list, g_list = param_grid["svc__C"], param_grid["svc__gamma"]
scores = grid.cv_results_["mean_test_score"].reshape(len(C_list), len(g_list))
im = axes[0].imshow(scores, cmap="viridis", aspect="auto")
axes[0].set_xticks(range(len(g_list)), [f"{g:g}" for g in g_list])
axes[0].set_yticks(range(len(C_list)), [f"{c:g}" for c in C_list])
axes[0].set_xlabel("gamma"); axes[0].set_ylabel("C")
axes[0].set_title("① 调参地形: 交叉验证准确率")
for i in range(len(C_list)):
    for j in range(len(g_list)):
        axes[0].text(j, i, f"{scores[i,j]:.2f}", ha="center", va="center",
                     color="white" if scores[i,j] < 0.95 else "black", fontsize=8)
fig.colorbar(im, ax=axes[0], fraction=0.046)

# (b) 混淆矩阵
cm = confusion_matrix(y_te, y_pred)
axes[1].imshow(cm, cmap="Blues")
axes[1].set_xticks([0,1], data.target_names); axes[1].set_yticks([0,1], data.target_names)
axes[1].set_xlabel("预测"); axes[1].set_ylabel("真实")
axes[1].set_title("② 测试集混淆矩阵")
for i in range(2):
    for j in range(2):
        axes[1].text(j, i, cm[i,j], ha="center", va="center", fontsize=13,
                     color="white" if cm[i,j] > cm.max()/2 else "black")

# (c) 把 30 维投到 2 维(PCA)后画 RBF 决策边界, 直观看一眼
sc = StandardScaler().fit(X_tr)
pca = PCA(n_components=2).fit(sc.transform(X_tr))
Z_tr, Z_te = pca.transform(sc.transform(X_tr)), pca.transform(sc.transform(X_te))
svc2 = SVC(kernel="rbf", C=grid.best_params_["svc__C"], gamma="scale").fit(Z_tr, y_tr)
xx, yy = np.meshgrid(np.linspace(Z_tr[:,0].min()-1, Z_tr[:,0].max()+1, 350),
                     np.linspace(Z_tr[:,1].min()-1, Z_tr[:,1].max()+1, 350))
P = svc2.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
axes[2].contourf(xx, yy, P, cmap=ListedColormap(["#f6d8c9", "#cfe3f7"]), alpha=0.9)
axes[2].contour(xx, yy, P, levels=[0.5], colors="#444441", linewidths=1.3)
axes[2].scatter(Z_te[:,0], Z_te[:,1], c=[coral if t==0 else blue for t in y_te],
                edgecolors="white", linewidths=0.6, s=30)
axes[2].set_xticks([]); axes[2].set_yticks([])
axes[2].set_title("③ PCA 2维投影上的 RBF 决策边界")

plt.tight_layout()
plt.savefig("svm_case.png", dpi=130, bbox_inches="tight")
print("\n图已保存. 测试集准确率: %.3f" % (y_pred == y_te).mean())

数据形状: (569, 30) | 类别: {'malignant': 212, 'benign': 357}
最佳参数: {'svc__C': 100, 'svc__gamma': 0.001} | 交叉验证准确率: 0.988

测试集表现:
               precision    recall  f1-score   support

   malignant       0.98      0.91      0.94        53
      benign       0.95      0.99      0.97        90

    accuracy                           0.96       143
   macro avg       0.96      0.95      0.95       143
weighted avg       0.96      0.96      0.96       143


图已保存. 测试集准确率: 0.958
